In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

# Historical Volatility

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/PredictionIpynb/HistoricalVolatilityLnReturn.ipynb"

In [ ]:
# Before visualizing, few matplotlib settings
mpl.rcParams['axes.labelsize'] = 'x-large'
mpl.rcParams['axes.labelpad'] = 5.5  # space between axis and label
mpl.rcParams['axes.titlesize'] = 'large'
mpl.rcParams['axes.titlepad'] = '20.0'

mpl.rcParams['legend.fontsize'] = 'x-large'

# mpl color settings
default_clrs = plt.rcParams['axes.prop_cycle'].by_key()['color']

# and seaborn settings
sns.set_style('darkgrid')

# Data Preprocessing for Neural Net

 ## Normalize data

In [ ]:
# Import packages
import os
import pickle

import numpy as np 
import pandas as pd 

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from scipy import stats
import scipy.optimize as optimize 

In [ ]:
from sklearn.model_selection import train_test_split
X = df
y = df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = False, stratify = None)

In [ ]:
# Normalize data

# Keep the original
train = X_train#train.copy()
test = X_test#test.copy()

# Variables that has to be normalized
to_normalize = ['vix_ln_return(t-1)',#'Historical Volatility Ln Return', 
                # 'neg', 'pos', 'neu', 'compound',
                'Polarity', 'Subjectivity',
                'Negative return', 'Positive return', 'Neutral return', 'Compound return', 'PressReleaseCount return'
                 ]
# Normalize
mean_train = train[to_normalize].mean()
std_train = train[to_normalize].std()

train[to_normalize] = (train[to_normalize]-mean_train)/std_train
test[to_normalize] = (test[to_normalize]-mean_train)/std_train

In [ ]:
# Select input variables
input_var =  ['vix_ln_return(t-1)',#'Historical Volatility Ln Return', 
                # 'neg', 'pos', 'neu', 'compound',
                'Polarity', 'Subjectivity',
                'Negative return', 'Positive return', 'Neutral return', 'Compound return', 'PressReleaseCount return'
                 ]
print(f'total {len(input_var)} variables')

# Subsample data
x_train = train[input_var]
x_test = test[input_var]
y_train = train['VIX return']
y_test = test['VIX return']

print('x_train', x_train.shape)
print('y_train', y_train.shape)
print('x_test', x_test.shape)
print('y_test', y_test.shape)

In [ ]:
TIME_STEPS = 4  #predict with 5 days (+1 because we already shifted 1 day in 1-3)

# function
def reshape(data, time_steps = TIME_STEPS):
    stack = [data.iloc[i:i+time_steps].to_numpy() for i in range(len(data)-time_steps)]
    reshaped = np.stack(stack, axis=0) 
    return reshaped

# reshape
x_train_val = reshape(x_train) # train data and validation data
y_train_val = reshape(y_train)
x_test = reshape(x_test)
y_test = reshape(y_test)

print('x_train + x_val', x_train_val.shape)
print('y_train + y_val', y_train_val.shape)
print('x_test', x_test.shape)
print('y_test', y_test.shape)

In [ ]:
# Seperate validation data from train data
i = int(len(x_train_val) * 0.7)

x_train = x_train_val[:i]
x_val = x_train_val[i:]
y_train = y_train_val[:i]
y_val = y_train_val[i:]

print('train data:', x_train.shape[0])
print('validation data:', x_val.shape[0])
print('test data:', x_test.shape[0])

5. Neural Network
In this section, we build a neural network which includes historical volatility as an input variable.
Other option-related variables are also included in order to predict implied volatility.

We will use a couple of different neural network.

Vanilla NN
LSTM
GRU
5-1. Hyperparameter tuning
Bayesian optimization will be applied using kerastuner.

In [ ]:
from kerastuner import BayesianOptimization

# Function 'model_builder' builds and compiles neural network with given hyperparameters

# vanilla nn
def nn_builder(hp):
    # hp
    UNITS = hp.Int('UNITS', min_value = 16, max_value = 128, step = 16)
    ACTIVATION_1 = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    ACTIVATION_2 = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    
    # model instance
    model = keras.Sequential([
        keras.layers.Flatten(),
        keras.layers.Dense(units = UNITS, activation = ACTIVATION_1),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = ACTIVATION_2)
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])    
    
    return model

# lstm
def lstm_builder(hp):
    # hp
    UNITS_1 = hp.Int('UNITS_1', min_value = 16, max_value = 128, step = 16)
    UNITS_2 = hp.Int('UNITS_2', min_value = 8, max_value = 32, step = 8)
    ACTIVATION = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    
    # model instance
    model = keras.Sequential([
        # input_shape = (time_steps, # of features)
        keras.layers.LSTM(units = UNITS_1, input_shape = (None, x_train.shape[2]), return_sequences = False), 
        keras.layers.Dense(UNITS_2, activation = ACTIVATION),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = 'linear')
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])
    
    return model

# gru
def gru_builder(hp):
    # hp
    UNITS_1 = hp.Int('UNITS_1', min_value = 16, max_value = 128, step = 16)
    UNITS_2 = hp.Int('UNITS_2', min_value = 8, max_value = 32, step = 8)
    ACTIVATION = hp.Choice('ACTIVATION',values = ['relu','softmax','linear'])
    
    # model instance
    model = keras.Sequential([
        # input_shape = (time_steps, # of features)
        keras.layers.GRU(units = UNITS_1, input_shape = (None, x_train.shape[2]), return_sequences = False),
        keras.layers.Dense(UNITS_2, activation = ACTIVATION),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = 'linear')
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])
    
    return model

In [ ]:
# Implement earlystopping with keras callback 

PATIENCE = 5 # number of epochs with no improvement after which training will be stopped.

Earlystopping = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              min_delta = 0.001,
                                              patience=PATIENCE, 
                                              mode='min', 
                                              restore_best_weights=True)

In [ ]:
# Search hyperparameters
SEED = 121

# NN
tuner_nn = BayesianOptimization(nn_builder,
                                objective = 'val_loss',
                                max_trials = 20,
                                seed = SEED,
                                directory = '/Users/Farid Srouji/Documents/kerastuner',
                                overwrite = True
                                )

tuner_nn.search(x_train, y_train, epochs=50, validation_data=(x_val, ), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_nn = tuner_nn.get_best_hyperparameters()[0]
model_nn = tuner_nn.hypermodel.build(besthp_nn)


# lstm
tuner_lstm = BayesianOptimization(lstm_builder,
                            objective = 'val_loss',
                            max_trials = 20,
                            seed = SEED,
                            directory = '/Users/Farid Srouji/Documents/kerastuner')

tuner_lstm.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_lstm = tuner_lstm.get_best_hyperparameters()[0]
model_lstm = tuner_lstm.hypermodel.build(besthp_lstm)


# gru
tuner_gru = BayesianOptimization(gru_builder,
                            objective = 'val_loss',
                            max_trials = 20,
                            seed = SEED,
                            directory = '/Users/Farid Srouji/Documents/kerastuner')

tuner_gru.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_gru = tuner_gru.get_best_hyperparameters()[0]
model_gru = tuner_gru.hypermodel.build(besthp_gru)

In [ ]:
besthp_nn

In [ ]:
# Colors designated to each model
palette = default_clrs[:4]
palette.append(default_clrs[6])

In [ ]:
# Ensure reproducible result
print(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Train model - NN
history = model_nn.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# Plot loss
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['val_loss'], label = 'val_loss', color=palette[2])
plt.legend()

# Loss
print('final loss:',history.history['loss'][-1])
print('final val_loss:',history.history['val_loss'][-1])

# aDDed

In [ ]:
# # summarize history for accuracy
# plt.plot(history.history['accuracy'])
# plt.plot(history.history['val_accuracy'])
# plt.title('model accuracy')
# plt.ylabel('accuracy')
# plt.xlabel('epoch')
# plt.legend(['train', 'test'], loc='upper left')
# plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
model_nn.summary()

In [ ]:
# Train model - LSTM
history = model_lstm.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# Plot loss
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['val_loss'], label = 'val_loss',color=palette[3])
plt.legend()
# Loss
print('final loss:',history.history['loss'][-1])
print('final val_loss:',history.history['val_loss'][-1])

In [ ]:
model_lstm.summary()

In [ ]:
# Train model - GRU
history = model_gru.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# Plot loss
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['val_loss'], label = 'val_loss',color=palette[4])
plt.legend()

# Loss
print('final loss:',history.history['loss'][-1])
print('final val_loss:',history.history['val_loss'][-1])

In [ ]:
model_gru.summary()

# Result

In [ ]:
# Data for comparison
target_date = test['Date'].iloc[TIME_STEPS:] 
target_VIX = test['VIX'].iloc[TIME_STEPS:]
# predictions_hv = test['Historical Volatility'].iloc[TIME_STEPS:]

# Function to scale back predictions
def scale_back(predictions):
    predictions = [prediction[0] for prediction in predictions] # unpack array

    # scale back
    mean_vol = train['VIX'].mean()
    std_vol = train['VIX'].std() 
    predictions = [prediction * std_vol + mean_vol for prediction in predictions]
    
    return predictions

# Predict
predictions_nn = scale_back(model_nn.predict(x_test))
predictions_lstm = scale_back(model_lstm.predict(x_test))
predictions_gru = scale_back(model_gru.predict(x_test))

# pd.Series-ization
predictions_nn = pd.Series(predictions_nn, index=target_VIX.index, name='predictions_nn')
predictions_lstm = pd.Series(predictions_lstm, index=target_VIX.index, name='predictions_lstm')
predictions_gru = pd.Series(predictions_gru, index=target_VIX.index, name='predictions_gru')

In [ ]:
label_date = train['Date']#.iloc[TIME_STEPS:] 
label_VIX = train['VIX']#.iloc[:TIME_STEPS]

In [ ]:
target_date

In [ ]:
# Plot
fig1, ax1 = plt.figure(figsize=(20,7)), plt.axes()
fig2, ax2 = plt.figure(figsize=(20,7)), plt.axes()
fig3, ax3 = plt.figure(figsize=(20,7)), plt.axes()
fig4, ax4 = plt.figure(figsize=(20,7)), plt.axes()

# label_date 
ax1.plot(label_date, label_VIX, label='VIX (train data)')
ax2.plot(label_date, label_VIX, label='VIX (train data)')
ax3.plot(label_date, label_VIX, label='VIX (train data)')
ax4.plot(label_date, label_VIX, label='VIX (train data)')

# target data
ax1.plot(target_date, target_VIX, label='VIX (target data)')
ax2.plot(target_date, target_VIX, label='VIX (target data)')
ax3.plot(target_date, target_VIX, label='VIX (target data)')
ax4.plot(target_date, target_VIX, label='VIX (target data)')

# predictions
# ax1.plot(target_date, predictions_hv, label='GARCH',color=palette[1])
ax2.plot(target_date, predictions_nn, label='GARCH-NN',color=palette[2])
ax3.plot(target_date, predictions_lstm, label='GARCH-LSTM',color=palette[3])
ax4.plot(target_date, predictions_gru, label='GARCH-GRU',color=palette[4])

# show
ax1.legend()
ax2.legend()
ax3.legend()
ax4.legend()
plt.show()

In [ ]:
# Plot
fig1, ax1 = plt.figure(figsize=(20,7)), plt.axes()
fig2, ax2 = plt.figure(figsize=(20,7)), plt.axes()
fig3, ax3 = plt.figure(figsize=(20,7)), plt.axes()
fig4, ax4 = plt.figure(figsize=(20,7)), plt.axes()

# target data
ax1.plot(target_date, target_VIX, label='VIX (target data)')
ax2.plot(target_date, target_VIX, label='VIX (target data)')
ax3.plot(target_date, target_VIX, label='VIX (target data)')
ax4.plot(target_date, target_VIX, label='VIX (target data)')

# predictions
# ax1.plot(target_date, predictions_hv, label='GARCH',color=palette[1])
ax2.plot(target_date, predictions_nn, label='GARCH-NN',color=palette[2])
ax3.plot(target_date, predictions_lstm, label='GARCH-LSTM',color=palette[3])
ax4.plot(target_date, predictions_gru, label='GARCH-GRU',color=palette[4])

# show
ax1.legend()
ax2.legend()
ax3.legend()
ax4.legend()
plt.show()

# MSE and MAE

In [ ]:
# Calculate error
def cal_MSE(predictions):
    errors = (predictions - target_VIX)**2
    return errors.mean()

def cal_MAE(predictions):
    errors = abs(predictions - target_VIX)
    return errors.mean()


# Print error
# print('GARCH')
# print('MSE:',cal_MSE(predictions_hv))
# print('MAE:',cal_MAE(predictions_hv))

print('GARCH-NN')
print('MSE:',cal_MSE(predictions_nn))
print('MAE:',cal_MAE(predictions_nn))

print('GARCH-LSTM')
print('MSE:',cal_MSE(predictions_lstm))
print('MAE:',cal_MAE(predictions_lstm))

print('GARCH-GRU')
print('MSE:',cal_MSE(predictions_gru))
print('MAE:',cal_MAE(predictions_gru))

# Number of best predictions

In [ ]:
# Compare day by day

# error list
# errors_hv = abs(target_VIX - predictions_hv)
errors_nn = abs(target_VIX - predictions_nn)
errors_lstm = abs(target_VIX - predictions_lstm)
errors_gru = abs(target_VIX - predictions_gru)

# count the best models
best_models = []
for i in range(len(errors_hv)):
    each_predictions = [errors_hv.iloc[i], errors_nn.iloc[i], errors_lstm.iloc[i], errors_gru.iloc[i]]
    best_models.append(each_predictions.index(min(each_predictions)))

best_counts = [best_models.count(i) for i in range(4)]


# plot
fig, ax = plt.figure(figsize=(10,7)), plt.axes()
ax.pie(best_counts, labels=['GARCH','GARCH-NN','GARCH-LSTM','GARCH-GRU'], autopct='%.0f%%', startangle=90, colors=palette[1:])
plt.title('Number of days in which it performed the best')
plt.show()

# Number of predicting the right direction (up or down)

In [ ]:
Names= ['GARCH','GARCH-NN','GARCH-LSTM','GARCH-GRU', 'Machine-Learning', 'LogisticRegression', 'Linear Regression', 'OLS','VAR']

In [ ]:
target_VIX

In [ ]:
direction_counts_all

In [ ]:
len(directions_VIX)

In [ ]:
# target_VIX.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/target_VIX.pkl")
# predictions_hv.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_hv.pkl")
predictions_nn.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_nn_return.pkl")
predictions_lstm.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_lstm_return.pkl")
predictions_gru.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_gru_return.pkl")

In [ ]:
# direction arrays
directions_VIX = target_VIX - target_VIX.shift()
directions_hv = predictions_hv - predictions_hv.shift()
directions_nn = predictions_nn - predictions_nn.shift()
directions_lstm = predictions_lstm - predictions_lstm.shift()
directions_gru = predictions_gru - predictions_gru.shift()

# multiplicate element-wise, and count the positivie elements
def count_pos(series):
    count = 0
    for x in series:
        if x>0:
            count += 1
    return count

direction_counts = [62,0,0,0,0]
direction_counts[1] = count_pos(directions_hv * directions_VIX)
direction_counts[2] = count_pos(directions_nn * directions_VIX)
direction_counts[3] = count_pos(directions_lstm * directions_VIX)
direction_counts[4] = count_pos(directions_gru * directions_VIX)

# plot
fig, ax = plt.figure(figsize=(10,7)), plt.axes()
ax.bar(direction_counts, color=palette)
plt.title('Number of days in which it predicted the right direction(up or down)')
plt.show()

# direction prediction accuracy
print('GARCH')
print(direction_counts[1] / direction_counts[0]*100,'%')
print('GARCH-NN')
print(direction_counts[2] / direction_counts[0]*100,'%')
print('GARCH-LSTM')
print(direction_counts[3] / direction_counts[0]*100,'%')
print('GARCH-GRU')
print(direction_counts[4] / direction_counts[0]*100,'%')


In [ ]:
direction_counts